# Ecosistema de procesamiento paralelo para datos masivos

## Módulo 3. Lección 2: Programación MapReduce con lenguajes de alto nivel: Hive

### Ejercicios resueltos 2

**IMPORTANTE**

Ejecuta las celdas en orden para evitar errores de ejecución. Esto garantiza que todos los archivos necesarios se creen y se ubiquen correctamente a lo largo del proceso.

#### Obtención de los datos

Utilizamos el dataset Convocatorias y Adjudicaciones

In [1]:
!mkdir -p /media/notebooks/argentina/

In [2]:
# Descargamos el fichero 
! wget https://github.com/Vega90/datasets/raw/main/convocatorias-2020.csv -O \
 /media/notebooks/argentina/convocatorias-2020.csv

--2024-08-28 10:02:41--  https://github.com/Vega90/datasets/raw/main/convocatorias-2020.csv
Resolving github.com (github.com)... 140.82.121.4
Connecting to github.com (github.com)|140.82.121.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/Vega90/datasets/main/convocatorias-2020.csv [following]
--2024-08-28 10:02:42--  https://raw.githubusercontent.com/Vega90/datasets/main/convocatorias-2020.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5675992 (5.4M) [text/plain]
Saving to: ‘/media/notebooks/argentina/convocatorias-2020.csv’

/media/notebooks/ar 100%[===================>]   5.41M  14.5MB/s    in 0.4s    

2024-08-28 10:02:43 (14.5 MB/s) - ‘/media/notebooks/argentina/convocatorias-2020.c

In [3]:
# Descargamos el fichero 
! wget https://github.com/Vega90/datasets/raw/main/adjudicaciones-2020.csv -O \
 /media/notebooks/argentina/adjudicaciones-2020.csv

--2024-08-28 10:02:43--  https://github.com/Vega90/datasets/raw/main/adjudicaciones-2020.csv
Resolving github.com (github.com)... 140.82.121.4
Connecting to github.com (github.com)|140.82.121.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/Vega90/datasets/main/adjudicaciones-2020.csv [following]
--2024-08-28 10:02:44--  https://raw.githubusercontent.com/Vega90/datasets/main/adjudicaciones-2020.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2659378 (2.5M) [text/plain]
Saving to: ‘/media/notebooks/argentina/adjudicaciones-2020.csv’

/media/notebooks/ar 100%[===================>]   2.54M  12.7MB/s    in 0.2s    

2024-08-28 10:02:44 (12.7 MB/s) - ‘/media/notebooks/argentina/adjudicaciones-2

In [4]:
# subimos el fichero a HDFS 
! hdfs dfs -mkdir /argentina/adjudicaciones
! hdfs dfs -put /media/notebooks/argentina/adjudicaciones-2020.csv /argentina/adjudicaciones
! hdfs dfs -ls /argentina/adjudicaciones

Found 1 items
-rw-r--r--   3 root supergroup    2659378 2024-08-28 10:02 /argentina/adjudicaciones/adjudicaciones-2020.csv


**Creación carpeta para los ejercicios**

In [5]:
import os
# creamos la carpeta para los ejercicios en local
! mkdir -p /media/notebooks/argentina/hive

# nos vamos a esa carpeta y comprobamos donde estamos
os.chdir("/media/notebooks/argentina/hive")
! pwd

/media/notebooks/argentina/hive


#### Dataframe de pandas para comprender los datos y comparar la salida

Utilizaremos un dataframe de pandas para ayudarnos a entender los datos y también podemos comprobar los ejercicios comparando la salida dada y la salida esperada.

In [2]:
import pandas as pd
# dataframe convocatorias
columnas = [
             'numero_procedimiento', 'nro_saf', 'Descripcion SAF', 'nro_ouc', 
             'descripcion_ouc','tipo-de_procedimiento', 'modalidad', 'apartado_directa', 
             'ejercicio', 'fecha_de_publicacion', 'fecha_de_apertura', 'etapa', 'alcance', 
             'nombre_del_procedimiento', 'objeto_del_procedimiento', 'monto_estimado', 'tipo_de_operacion'
           ]

df_convocatorias = pd.read_csv('/media/notebooks/argentina/convocatorias-2020.csv', names=columnas)

# dataframe adjudicaciones
columnas = [
             'numero_procedimiento', 'nro_saf', 'nro_ouc', 'unidad_ejecutora',  
             'ejercicio_adjudicacion', 'fecha_adjudicacion', 'rubros', 
             'cuit', 'descripcion_proveedor', 'documento_contractual',
             'tipo', 'monto', 'moneda', 'fecha_perfeccionamiento_oc'
           ]

df_adjudicaciones = pd.read_csv('/media/notebooks/argentina/adjudicaciones-2020.csv', names=columnas)

# mostramos los primeros registros 
df_adjudicaciones.head(3)

,numero_procedimiento,nro_saf,nro_ouc,unidad_ejecutora,ejercicio_adjudicacion,fecha_adjudicacion,rubros,cuit,descripcion_proveedor,documento_contractual,tipo,monto,moneda,fecha_perfeccionamiento_oc
0,84/22-1322-LPR20,374,84,84/22 - Comando de Brigada de Monte XII,2020,05/11/2020 12:24:00 p.m.,ALIMENTOS$,20-18053177-8,Federico Pablo Campolongo,84/22-4671-OC20,Original,27164,Peso Argentino,05/11/2020 12:00:00 a.m.
1,340/1-0001-LPR20,376,340,340/001 - Dirección General de Administración ...,2020,28/10/2020 09:58:03 a.m.,CERRAJERIA$ELECTRICIDAD Y TELEFONIA$FERRETERIA...,30-71154458-1,Trigemios S.R.L,340/1-1062-OC20,Original,5610,Peso Argentino,28/10/2020 12:00:00 a.m.
2,38/13-0724-LPR20,379,38,38/13 - Escuela de Tecnicas y Tacticas Navales,2020,30/09/2020 12:31:19 a.m.,CERRAJERIA$EQUIPOS$FERRETERIA$HERRAMIENTAS$IND...,20-290459347-7,DIEGO JULIO PEREZ,38/13-0511-OCA20,SPR,1050,Peso Argentino,30/09/2020 12:47:24 a.m.


#### Creación de la tabla Adjudicaciones.

Lo primero que debemos hacer es crear la tabla con la que realizaremos los ejercicios de esta sección. Para ello es necesario conocer los tipos que tenemos de datos y crear el fichero de consulta adecuado. La tabla *Convocatorias* también debe estar creada en Hive.

In [7]:
%%writefile tabla-adjudicaciones.hql
CREATE EXTERNAL TABLE adjudicaciones (
     numero_procedimiento STRING,
     nro_saf INT,
     nro_uoc INT,
     unidad_ejecutora STRING,
     ejercicio_adjudicacion INT,
     fecha_adjudicacion STRING,
     rubros ARRAY <STRING>,
     cuit STRING,
     descripcion_proveedor STRING,
     documento_contractual STRING,
     tipo STRING,
     monto FLOAT,
     moneda STRING,
     fecha_perfeccionamiento_oc STRING
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ','
COLLECTION ITEMS TERMINATED BY '$'
STORED AS TEXTFILE
LOCATION '/argentina/adjudicaciones/';

Writing tabla-adjudicaciones.hql


In [8]:
# ejecucion del fichero hql mediante beeline
!beeline -u "jdbc:hive2://" -f tabla-adjudicaciones.hql

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Connecting to jdbc:hive2://
Hive Session

In [9]:
#mostramos las tres primeas filas
!beeline -u "jdbc:hive2://" -e \
"SELECT numero_procedimiento, unidad_ejecutora, fecha_adjudicacion, monto \
FROM adjudicaciones \
LIMIT 3;"

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Connecting to jdbc:hive2://
Hive Session

#### **Ejercicio 07.** Unión de tablas con Apache Hive.

En este ejercicio vamos a realizar la unión de dos tablas para crear una tercera con la que podremos realizar consultas y análisis más profundos. Necesitaremos el campo común a ambas tablas numero_procedimiento y que actúe como clave.

Necesitaremos el campo común a ambas tablas numero_procedimiento. A partir de la consulta, crearemos una nueva tabla que sea la unión de las dos cuando haya coincidencias en ambas tablas (INNER JOIN). Aprovecharemos para ordenar algunas columnas, con el fin de que quede más ordenado y claro.

A continuación, también dejamos el código para crear la tabla Convocatorias, por si no estuvieses creada en Hive.

In [10]:
%%writefile tabla-convocatorias.hql
CREATE EXTERNAL TABLE convocatorias (
     numero_procedimiento STRING,
     nro_saf INT,
     descripcion_saf STRING,
     nro_uoc INT,
     descripcion_uoc STRING,
     tipo_de_procedimiento STRING,
     modalidad STRING,
     apartado_directa STRING,
     ejercicio INT,
     fecha_de_publicacion STRING,
     fecha_de_apertura STRING,
     etapa STRING,
     alcance STRING,
     nombre_del_procedimiento STRING,
     objeto_del_procedimiento STRING,
     monto_estimado FLOAT,
     tipo_de_operacion STRING
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ','
STORED AS TEXTFILE
LOCATION '/argentina/convocatorias/'
TBLPROPERTIES ("skip.header.line.count"="0");

Overwriting tabla-convocatorias.hql


In [11]:
%%writefile tabla-nueva.hql
CREATE TABLE tabla_union AS
SELECT
     conv.numero_procedimiento,
     conv.nro_saf AS nro_saf,
     conv.descripcion_saf AS descripcion_saf,
     conv.nro_uoc AS nro_uoc,
     conv.descripcion_uoc AS descripcion_uoc,
     conv.tipo_de_procedimiento AS tipo_de_procedimiento,
     conv.modalidad AS modalidad,
     conv.apartado_directa AS apartado_directa,
     conv.ejercicio AS ejercicio,
     adj.ejercicio_adjudicacion AS ejercicio_adjudicacion,
     conv.fecha_de_publicacion AS fecha_de_publicacion,
     adj.fecha_adjudicacion AS fecha_adjudicacion,
     conv.fecha_de_apertura AS fecha_de_apertura,
     conv.etapa AS etapa,
     conv.alcance AS alcance,
     conv.nombre_del_procedimiento AS nombre_del_procedimiento,
     conv.objeto_del_procedimiento AS objeto_del_procedimiento,
     conv.monto_estimado AS monto_estimado,
     adj.monto AS monto_adjudicado,
     adj.moneda AS moneda_adj,
     conv.tipo_de_operacion AS tipo_de_operacion,
     adj.unidad_ejecutora AS unidad_ejecutora_adj,
     adj.rubros AS rubros,
     adj.cuit AS cuit,
     adj.descripcion_proveedor AS descripcion_proveedor,
     adj.documento_contractual AS documento_contractual
FROM convocatorias AS conv
INNER JOIN adjudicaciones AS adj
ON  conv.numero_procedimiento = adj.numero_procedimiento;

Writing tabla-nueva.hql


In [12]:
# ejecucion del fichero hql mediante beeline
!beeline -u "jdbc:hive2://" -f tabla-convocatorias.hql

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Connecting to jdbc:hive2://
Hive Session

In [13]:
# ejecucion del fichero hql mediante beeline
!beeline -u "jdbc:hive2://" -f tabla-nueva.hql

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Connecting to jdbc:hive2://
Hive Session

In [14]:
# mostramos las tres primeras filas 
!beeline -u "jdbc:hive2://" -e \
"SELECT numero_procedimiento, nro_saf, ejercicio_adjudicacion, ejercicio \
FROM tabla_union \
LIMIT 3;"

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Connecting to jdbc:hive2://
Hive Session

In [15]:
# contamos el numero de registros
!beeline -u "jdbc:hive2://" -e \
"SELECT Count(*) as num_registros \
FROM tabla_union;"

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Connecting to jdbc:hive2://
Hive Session

**COMPARAMOS LA SALIDA ESPERADA CON LA OBTENIDA**

In [3]:
# realizamos la union
df_union = pd.merge( 
                     df_adjudicaciones, df_convocatorias,
                     on=['numero_procedimiento'],
                     how='inner', suffixes=('_ad', '_conv')
                   ).sort_values(by=('numero_procedimiento'))
df_union[['numero_procedimiento','nro_saf_conv','ejercicio_adjudicacion','ejercicio']].head(3)

,numero_procedimiento,nro_saf_conv,ejercicio_adjudicacion,ejercicio
6199,101-0001-CDI20,622,2020,2020
5587,101-0002-CDI20,622,2020,2020
4274,101-0002-LPR20,622,2020,2020


In [17]:
# numero total de registros
df_union.shape[0]

8633

#### **Ejercicio 08.** Top N con Apache Hive.

El ejercicio número ocho consiste en extraer información sobre la tabla unión que hemos realizado. Vamos a calcular el top 5, de aquellos procedimientos que salieron más caros de lo esperado. Es decir, el monto estimado fue menor que el monto adjudicado.

Para realizar esta consulta necesitamos los campos monto_estimado, monto_adjudicado y numero_procedimiento, este último campo lo pondremos para ubicar el procedimiento. Para ordenar los resultados y quedarnos con los cinco primeros, utilizaremos las cláusulas ORDER BY campo DESC, y LIMIT numero.

In [5]:
# top 5 procedimientos
!beeline -u "jdbc:hive2://" -e \
"SELECT \
    numero_procedimiento, monto_estimado , monto_adjudicado, \
    (monto_estimado - monto_adjudicado) AS monto_diferencia \
FROM tabla_union \
ORDER BY monto_diferencia ASC \
LIMIT 5;"

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Connecting to jdbc:hive2://
Hive Session

In [4]:
# comprobamos en el dataframe
import warnings
warnings.filterwarnings('ignore')

# calculamos la diferencia
df_diferencias = df_union[['numero_procedimiento', 'monto_estimado', 'monto']]
df_diferencias['monto_diferencia'] = (df_diferencias['monto_estimado'] - df_diferencias['monto'])
df_diferencias.sort_values(by=('monto_diferencia'), ascending = True).head(5)

,numero_procedimiento,monto_estimado,monto,monto_diferencia
7831,21-0012-CDI20,94505542.86,9450554286,-9.356049e+09
1931,34-0008-LPU20,85782730.92,8749441686,-8.663659e+09
2618,483-0003-LPU20,71842500.00,7998663854,-7.926821e+09
4097,39-0013-CDI20,77605709.53,7760570953,-7.682965e+09
1987,68-0013-LPU20,80290000.00,7347119393,-7.266829e+09


#### **Ejercicio 09.** Particiones en Apache Hive.

Este ejercicio va a utilizar tres datasets sobre la edad de los alumnos en las escuelas de Argentina.

Los campos son:

1.  ID: Identificación de la escuela.
2.  Provincia: Provincia donde se ubica la escuela.
3.  Sector: Sector de la escuela. Estatal, privado, etc.
4.  Ámbito: Ámbito donde se ubica la escuela. Rural, Urbano, etc.
5.  Grado: Curso de los alumnos

A parte de estos campos existen los campos propios de la edad: 0 años, 1 año, 2 años, etc. Y otros campos con rangos: menos de 18 años, 6 años y más, de 25 a 29 años, etc.

Vamos a descargar los tres ficheros, descomprimirlos y subirlos a HDFS para poder crear desde allí la tabla con las particiones.

In [7]:
from zipfile import ZipFile

# Descargamos el fichero y lo guardamos donde queremos
! wget https://github.com/Vega90/datasets/raw/main/argentina_alumnos_por_edad.zip \
-O /media/notebooks/argentina/argentina_alumnos_por_edad.zip

# Descomprimimos el fichero zip
zip = ZipFile('/media/notebooks/argentina/argentina_alumnos_por_edad.zip')
zip.extractall('/media/notebooks/argentina/')
zip.close()

# Eliminamos el fichero zip
! rm /media/notebooks/argentina/argentina_alumnos_por_edad.zip

--2024-08-28 11:35:40--  https://github.com/Vega90/datasets/raw/main/argentina_alumnos_por_edad.zip
Resolving github.com (github.com)... 140.82.121.4
Connecting to github.com (github.com)|140.82.121.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/Vega90/datasets/main/argentina_alumnos_por_edad.zip [following]
--2024-08-28 11:35:41--  https://raw.githubusercontent.com/Vega90/datasets/main/argentina_alumnos_por_edad.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6872440 (6.6M) [application/zip]
Saving to: ‘/media/notebooks/argentina/argentina_alumnos_por_edad.zip’

/media/notebooks/ar 100%[===================>]   6.55M  19.2MB/s    in 0.3s    

2024-08-28 11:35:42 (19.2 MB/s) - ‘/media/not

In [21]:
# subimos el fichero a HDFS 
! hdfs dfs -mkdir /argentina/alumnos
! hdfs dfs -mkdir /argentina/alumnos/2021
! hdfs dfs -mkdir /argentina/alumnos/2022
! hdfs dfs -mkdir /argentina/alumnos/2023
! hdfs dfs -put /media/notebooks/argentina/2021_alumnos_por_edad.csv /argentina/alumnos/2021
! hdfs dfs -put /media/notebooks/argentina/2022_alumnos_por_edad.csv /argentina/alumnos/2022
! hdfs dfs -put /media/notebooks/argentina/2023_alumnos_por_edad.csv /argentina/alumnos/2023

In [22]:
! hdfs dfs -ls /argentina/alumnos/2023

Found 1 items
-rw-r--r--   3 root supergroup   24548268 2024-08-28 10:09 /argentina/alumnos/2023/2023_alumnos_por_edad.csv


Ahora vamos a crear una tabla alumnos_argentina que contenga los años 2021, 2022 y 2023, pero con los datos particionados.

Para ello debemos crear la tabla con los campos anteriores y añadir una cláusula PARTITIONED BY, con el año como campo para la partición. Además de esto omitiremos la cláusula de la localización ya que cargaremos los datos posteriormente.

In [23]:
%%writefile tabla-alumnos.hql
CREATE TABLE IF NOT EXISTS alumnos_argentina (
     ID STRING, provincia STRING, sector STRING,
     ambito STRING, grado STRING,
     0_year INT, 1_year INT, 2_year INT,
     3_year INT, 4_year INT, 5_year INT,
     6_year INT, 7_year INT, 8_year INT,
     9_year INT, 10_year INT, 11_year INT,
     12_year INT, 13_year INT, 14_year INT,
     15_year INT, 16_year INT, 17_year INT,
     18_year INT, 19_year INT, 20_year INT,
     21_year INT, 22_year INT, 23_year INT,
     24_year INT, 25_year INT, 26_year INT,
     27_year INT, 28_year INT, 29_year INT,
     less_18_year INT,  11_year_and_less INT,
     6_year_and_more INT, 18_year_and_more INT,
     25_year_and_more INT, 30_year_and_more INT, 
     20_to_24_year INT, 25_to_29_year INT
)
PARTITIONED BY (year INT)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ';'
STORED AS TEXTFILE;

Writing tabla-alumnos.hql


In [24]:
# creación tabla alumnos
!beeline -u "jdbc:hive2://" -f tabla-alumnos.hql

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Connecting to jdbc:hive2://
Hive Session

In [25]:
# carga datos 2021 
!beeline -u "jdbc:hive2://" -e \
"LOAD DATA INPATH '/argentina/alumnos/2021/2021_alumnos_por_edad.csv'  \
INTO TABLE alumnos_argentina \
PARTITION (year=2021);"

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Connecting to jdbc:hive2://
Hive Session

In [26]:
# carga datos 2022 
!beeline -u "jdbc:hive2://" -e \
"LOAD DATA INPATH '/argentina/alumnos/2022/2022_alumnos_por_edad.csv'  \
INTO TABLE alumnos_argentina \
PARTITION (year=2022);"

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Connecting to jdbc:hive2://
Hive Session

In [27]:
# carga datos 2023 
!beeline -u "jdbc:hive2://" -e \
"LOAD DATA INPATH '/argentina/alumnos/2023/2023_alumnos_por_edad.csv'  \
INTO TABLE alumnos_argentina \
PARTITION (year=2023);"

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Connecting to jdbc:hive2://
Hive Session

In [28]:
# mostramos las particiones
!beeline -u "jdbc:hive2://" -e \
"SHOW PARTITIONS alumnos_argentina;"

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Connecting to jdbc:hive2://
Hive Session

In [29]:
# mostramos las cinco filas primeras de 2021
!beeline -u "jdbc:hive2://" -e \
"SELECT  \
     ID, provincia, sector, ambito, grado, \
     6_year, 7_year, 8_year, 9_year \
FROM alumnos_argentina \
WHERE year=2021 \
LIMIT 5;"

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Connecting to jdbc:hive2://
Hive Session

In [8]:
import pandas as pd
# columnas de los tres ficheros
columnas = [
             'ID', 'provincia', 'sector', 'ambito', 'grado', 
             '0_year', '1_year', '2_year', '3_year', '4_year',
             '5_year', '6_year', '7_year', '8_year', '9_year',
             '10_year', '11_year', '12_year', '13_year', '14_year',
             '15_year', '16_year', '17_year', '18_year', '19_year',
             '20_year', '21_year', '22_year', '23_year', '24_year',
             '25_year', '26_year', '27_year', '28_year', '29_year',
             'less_18_year', '11_year_and_less', '6_year_and_more',
             '18_year_and_more', '25_year_and_more', '30_year_and_more',
             'cuit', 'descripcion_proveedor', 'documento_contractual',
             '20_to_24_year', '25_to_29_year'
           ]

ruta_carpeta = '/media/notebooks/argentina'
df_2021 = pd.read_csv(ruta_carpeta + '/2021_alumnos_por_edad.csv', names=columnas, delimiter=';')
df_2022 = pd.read_csv(ruta_carpeta + '/2022_alumnos_por_edad.csv', names=columnas, delimiter=';')
df_2023 = pd.read_csv(ruta_carpeta + '/2023_alumnos_por_edad.csv', names=columnas, delimiter=';')

#añadimos los años a cada columna
df_2021['year'] = 2021
df_2022['year'] = 2022
df_2023['year'] = 2023

# mostramos los primeros registros 
df_2021[['ID', 'provincia', 'sector', 'ambito', 'grado', '6_year', '7_year', '8_year', '9_year']].head(5)

,ID,provincia,sector,ambito,grado,6_year,7_year,8_year,9_year
0,20004002000400,Mendoza,Estatal,Urbano,1 grado,34.0,NaN,NaN,NaN
1,20004002000400,Mendoza,Estatal,Urbano,2 grado,NaN,43.0,NaN,NaN
2,20004002000400,Mendoza,Estatal,Urbano,3 grado,NaN,NaN,31.0,2.0
3,20004002000400,Mendoza,Estatal,Urbano,4 grado,NaN,NaN,NaN,41.0
4,20004002000400,Mendoza,Estatal,Urbano,5 grado,NaN,NaN,NaN,NaN


#### **Ejercicio 10.**

Para terminar esta sección vamos a realizar una consulta más compleja que utilice las tres particiones y realice una comparativa entre los alumnos de 11 años entre 2021 y 2023. Debemos tener en cuenta las siguientes consideraciones para realizar la consulta:

*  **WITH alumnos_year**: Agrupa los alumnos matriculados con 11 años por año en una tabla temporal.
*  **LAG**: La función LAG obtiene el valor de los alumnos de 11 años del año anterior de la tabla temporal.

In [31]:
# comparación entre los tres años para alumnos 11 años
!beeline -u "jdbc:hive2://" -e \
"WITH alumnos_year AS ( \
    SELECT  year, \
            SUM(11_year) AS alumnos_11_year \
    FROM alumnos_argentina  \
    WHERE year IN (2021, 2022, 2023) \
    GROUP BY year \
) \
SELECT  year, alumnos_11_year, \
    LAG(alumnos_11_year, 1) OVER (ORDER BY year) AS alumnos_11_anterior, \
    alumnos_11_year - (LAG(alumnos_11_year, 1) OVER (ORDER BY year)) AS diferencia  \
FROM alumnos_year \
ORDER BY year;"

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Connecting to jdbc:hive2://
Hive Session

In [9]:
# comprobamos en el dataframe
df_combinado = pd.concat([df_2021, df_2022, df_2023], ignore_index=True)

# agrupamos y sumamos los alumnos con 11 años por año
df_comparativa = df_combinado.groupby('year')['11_year'].sum().reset_index()

# realizamos la comparativa
df_comparativa['alumnos_11_anterior'] = df_comparativa['11_year'].shift(1)
df_comparativa['diferencia'] = df_comparativa['11_year'] - df_comparativa['alumnos_11_anterior']
df_comparativa

,year,11_year,alumnos_11_anterior,diferencia
0,2021,746113.0,NaN,NaN
1,2022,751849.0,746113.0,5736.0
2,2023,738572.0,751849.0,-13277.0
